# Translation Evaluation — Marathi ↔ Kannada via English Pivot
**Run this on Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)**

## Steps:
1. Upload `translation_test_100.json` when prompted
2. Run all cells in order
3. Download `translation_results.json` at the end

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers sacrebleu bert-score sentencepiece protobuf torch

In [ ]:
# ── Cell 2: Upload test data ───────────────────────────────────────────────────
from google.colab import files
print('Upload translation_test_100.json')
uploaded = files.upload()
print('✅ File uploaded!')

In [ ]:
# ── Cell 3: Load data + model ──────────────────────────────────────────────────
import json, time, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load test data
test_data = json.load(open('translation_test_100.json'))
print(f'Loaded {len(test_data)} test sentences')

# Detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Load NLLB model
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print('✅ Model loaded!')

LANG_CODES = {
    'marathi':  'mar_Deva',
    'kannada':  'kan_Knda',
    'hindi':    'hin_Deva',
    'english':  'eng_Latn',
}

In [ ]:
# ── Cell 4: Translation helper ─────────────────────────────────────────────────
def translate_batch(texts, src_lang, tgt_lang, batch_size=16):
    src_code = LANG_CODES[src_lang]
    tgt_code = LANG_CODES[tgt_lang]
    tokenizer.src_lang = src_code
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=256).to(device)
        forced_bos = tokenizer.convert_tokens_to_ids(tgt_code)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                 max_new_tokens=256, num_beams=4)
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        results.extend(decoded)
        print(f'  {min(i+batch_size, len(texts))}/{len(texts)} translated', end='\r')
    print()
    return results

print('✅ Translation helper ready')

In [ ]:
# ── Cell 5: Run all 4 experiments ─────────────────────────────────────────────
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer
import numpy as np

bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF()

en_refs  = [x['english']  for x in test_data]
mr_sents = [x['marathi']  for x in test_data]
kn_refs  = [x['kannada']  for x in test_data]

results = {}

# ── Experiment A: MR → EN → KN ──────────────────────────────────────────────
print('\n[Experiment A] Pivot: Marathi → English → Kannada')
t0 = time.time()
mr_to_en = translate_batch(mr_sents, 'marathi', 'english')
en_to_kn = translate_batch(mr_to_en, 'english', 'kannada')
_, _, bs = bert_score(en_to_kn, kn_refs, lang='kn', verbose=False)
results['mr_to_kn_pivot'] = {
    'corpus_bleu':     bleu_metric.corpus_score(en_to_kn, [kn_refs]).score,
    'corpus_chrf':     chrf_metric.corpus_score(en_to_kn, [kn_refs]).score,
    'bertscore_f1':    round(bs.mean().item(), 4),
    'n': len(en_to_kn),
    'time_seconds':    round(time.time()-t0, 1)
}
print(f"  BLEU={results['mr_to_kn_pivot']['corpus_bleu']:.2f}  chrF={results['mr_to_kn_pivot']['corpus_chrf']:.2f}  BERTScore={results['mr_to_kn_pivot']['bertscore_f1']}")

# ── Experiment B: KN → EN → MR ──────────────────────────────────────────────
print('\n[Experiment B] Pivot: Kannada → English → Marathi')
t0 = time.time()
kn_to_en = translate_batch(kn_refs,  'kannada', 'english')
en_to_mr = translate_batch(kn_to_en, 'english', 'marathi')
_, _, bs = bert_score(en_to_mr, mr_sents, lang='mr', verbose=False)
results['kn_to_mr_pivot'] = {
    'corpus_bleu':     bleu_metric.corpus_score(en_to_mr, [mr_sents]).score,
    'corpus_chrf':     chrf_metric.corpus_score(en_to_mr, [mr_sents]).score,
    'bertscore_f1':    round(bs.mean().item(), 4),
    'n': len(en_to_mr),
    'time_seconds':    round(time.time()-t0, 1)
}
print(f"  BLEU={results['kn_to_mr_pivot']['corpus_bleu']:.2f}  chrF={results['kn_to_mr_pivot']['corpus_chrf']:.2f}  BERTScore={results['kn_to_mr_pivot']['bertscore_f1']}")

# ── Experiment C: EN → MR → EN round-trip ───────────────────────────────────
print('\n[Experiment C] Round-trip: English → Marathi → English')
t0 = time.time()
en_to_mr_fwd  = translate_batch(en_refs,       'english', 'marathi')
mr_back_en    = translate_batch(en_to_mr_fwd,  'marathi', 'english')
sem_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
e1 = sem_model.encode(en_refs,   convert_to_tensor=True)
e2 = sem_model.encode(mr_back_en, convert_to_tensor=True)
cos_sim = torch.nn.functional.cosine_similarity(e1, e2).mean().item()
results['round_trip_en_mr_en'] = {
    'cosine_similarity': round(cos_sim, 4),
    'n': len(en_refs),
    'time_seconds': round(time.time()-t0, 1),
    'interpretation': 'Higher = more meaning preserved'
}
print(f"  Round-trip cosine similarity = {cos_sim:.4f}")

# ── Experiment D: EN → KN direct ────────────────────────────────────────────
print('\n[Experiment D] Direct: English → Kannada')
t0 = time.time()
en_to_kn_direct = translate_batch(en_refs, 'english', 'kannada')
_, _, bs = bert_score(en_to_kn_direct, kn_refs, lang='kn', verbose=False)
results['en_to_kn_direct'] = {
    'corpus_bleu':     bleu_metric.corpus_score(en_to_kn_direct, [kn_refs]).score,
    'corpus_chrf':     chrf_metric.corpus_score(en_to_kn_direct, [kn_refs]).score,
    'bertscore_f1':    round(bs.mean().item(), 4),
    'n': len(en_to_kn_direct),
    'time_seconds':    round(time.time()-t0, 1)
}
print(f"  BLEU={results['en_to_kn_direct']['corpus_bleu']:.2f}  chrF={results['en_to_kn_direct']['corpus_chrf']:.2f}  BERTScore={results['en_to_kn_direct']['bertscore_f1']}")

print('\n✅ All experiments done!')

In [ ]:
# ── Cell 6: Save and download results ─────────────────────────────────────────
with open('translation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved translation_results.json')

# Print summary table
print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
for exp, r in results.items():
    if 'corpus_bleu' in r:
        print(f"{exp:25s}  BLEU={r['corpus_bleu']:.2f}  chrF={r['corpus_chrf']:.2f}  BERTScore={r['bertscore_f1']}")
    else:
        print(f"{exp:25s}  cosine={r['cosine_similarity']}")

# Download the file
from google.colab import files
files.download('translation_results.json')
print('\n⬇️ translation_results.json downloaded!')